# HotpotQA Cross-Encoder Evaluation

这个 notebook 复用现有的 HotpotQA span 扫描缓存，只针对输入的 span 读取相关文本并做 cross-encoder 预测。

- 不生成 embedding。
- 候选定义来自 `search_wikidata(span, limit=5, include_detailed_description=True, detailed_description_sentences=3, drop_missing_detailed_description=True)`。
- 如果 `search_wikidata` 没有返回候选定义，直接报错并停止。
- 输出展示原始文本上下文以及预测分数最高的定义。


In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Working directory: {REPO_ROOT}")


Working directory: /home/xiaoyue/LiteSemRAG


In [2]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import requests
from IPython.display import display
from sentence_transformers import CrossEncoder

from text_processing import normalize_text
from wikidata_utils import fetch_detailed_descriptions_for_entities


In [3]:
HOTPOT_SCAN_STORE_PATH = Path("hotpot_QA_qwen_scan_cache/hotpot_phrase_token_scan_store_qwen_scan_v1_all.pkl")
DEFAULT_MODEL_NAME = "cross-encoder/nli-deberta-v3-base"
DEFAULT_BATCH_SIZE = 32


def load_hotpot_scan_store(scan_store_path: Path = HOTPOT_SCAN_STORE_PATH):
    if not scan_store_path.exists():
        raise FileNotFoundError(
            f"HotpotQA scan store not found at {scan_store_path}. Please prepare the scan cache first."
        )

    with scan_store_path.open("rb") as handle:
        store = pickle.load(handle)

    print(f"Loaded scan-only store from {scan_store_path}")
    print(store["stats"])
    return store


embedding_store = load_hotpot_scan_store()


Loaded scan-only store from hotpot_QA_qwen_scan_cache/hotpot_phrase_token_scan_store_qwen_scan_v1_all.pkl
{'num_documents': 66581, 'num_unique_terms': 634624, 'num_phrase_occurrences': 1007780, 'num_token_occurrences': 584999, 'num_total_occurrences': 1592779}


In [4]:
def lookup_records(store, query_text, kind=None, include_text=True, include_cleaned_text=True):
    normalized_query = normalize_text(query_text.strip())
    records = list(store["index"].get(normalized_query, []))

    if kind is not None:
        records = [record for record in records if record["kind"] == kind]

    if not include_text and not include_cleaned_text:
        return records

    enriched_records = []
    for record in records:
        item = dict(record)
        document_idx = item["document_idx"]
        if include_text:
            item["text"] = store["documents"][document_idx]["text"]
        if include_cleaned_text:
            item["cleaned_text"] = store["cleaned_documents"][document_idx]
        enriched_records.append(item)
    return enriched_records


def extract_sentence_context(cleaned_text, span):
    start_char, end_char = span
    left_boundary = max(
        cleaned_text.rfind(".", 0, start_char),
        cleaned_text.rfind("!", 0, start_char),
        cleaned_text.rfind("?", 0, start_char),
    )
    right_candidates = [
        cleaned_text.find(".", end_char),
        cleaned_text.find("!", end_char),
        cleaned_text.find("?", end_char),
    ]
    right_candidates = [idx for idx in right_candidates if idx != -1]

    context_start = 0 if left_boundary == -1 else left_boundary + 1
    context_end = len(cleaned_text) if not right_candidates else min(right_candidates) + 1
    context_raw = cleaned_text[context_start:context_end]

    if not context_raw.strip():
        context_start = max(0, start_char - 120)
        context_end = min(len(cleaned_text), end_char + 120)
        context_raw = cleaned_text[context_start:context_end]

    left_trim = len(context_raw) - len(context_raw.lstrip())
    context_text = context_raw.strip()

    local_start = start_char - context_start - left_trim
    local_end = end_char - context_start - left_trim
    local_start = max(0, min(local_start, len(context_text)))
    local_end = max(local_start, min(local_end, len(context_text)))

    return {
        "context_text": context_text,
        "local_span": (local_start, local_end),
    }


def build_hotpot_prompt(record, query_text, mark_target=False, left_marker="[TGT]", right_marker="[/TGT]"):
    context_info = extract_sentence_context(record["cleaned_text"], record["span"])
    context_text = context_info["context_text"]
    local_start, local_end = context_info["local_span"]
    prompt_context = context_text

    if mark_target:
        prompt_context = (
            f"{context_text[:local_start]}{left_marker} {context_text[local_start:local_end]} {right_marker}{context_text[local_end:]}"
        )

    prompt_text = (
        f"Sentence: {prompt_context}\n"
        f"Target word: {query_text.strip()}\n\n"
        f'Question: What does "{query_text.strip()}" mean in this sentence?'
    )

    return {
        "context_text": context_text,
        "matched_text": context_text[local_start:local_end],
        "local_span": (local_start, local_end),
        "prompt_text": prompt_text,
    }


In [5]:
WIKIDATA_API_URL = "https://www.wikidata.org/w/api.php"
DEFAULT_HEADERS = {
    "User-Agent": "WikidataExplorerNotebook/1.0 (https://www.wikidata.org/)"
}


def _safe_get_json(url, params=None, headers=None, timeout=30):
    try:
        response = requests.get(url, params=params, headers=headers or DEFAULT_HEADERS, timeout=timeout)
        response.raise_for_status()
        return response.json()
    except requests.RequestException as exc:
        print(f"Request failed: {exc}")
        return {}
    except ValueError as exc:
        print(f"Invalid JSON response: {exc}")
        return {}


def _coerce_aliases_for_search(value):
    if value is None:
        return ""
    if isinstance(value, list):
        return ", ".join(str(v) for v in value if v is not None)
    if isinstance(value, str):
        return value
    return str(value)


def search_wikidata(
    term,
    language="en",
    limit=10,
    exact_match_text=False,
    include_detailed_description=False,
    drop_missing_detailed_description=False,
    detailed_description_sentences=3,
):
    if not isinstance(term, str) or not term.strip():
        print("Please provide a non-empty search term.")
        columns = ["id", "label", "description", "match_text", "aliases", "concepturi"]
        if include_detailed_description:
            columns.insert(3, "detailed_description")
        return pd.DataFrame(columns=columns)

    normalized_term = term.strip()

    params = {
        "action": "wbsearchentities",
        "format": "json",
        "language": language,
        "uselang": language,
        "search": normalized_term,
        "limit": int(limit),
    }

    data = _safe_get_json(WIKIDATA_API_URL, params=params)
    items = data.get("search", []) if isinstance(data, dict) else []

    rows = []
    for item in items:
        match = item.get("match") if isinstance(item, dict) else None
        match_text = ""
        if isinstance(match, dict):
            match_text = match.get("text", "")

        row = {
            "id": item.get("id", ""),
            "label": item.get("label", ""),
            "description": item.get("description", ""),
            "match_text": match_text,
            "aliases": _coerce_aliases_for_search(item.get("aliases")),
            "concepturi": item.get("concepturi", ""),
        }
        rows.append(row)

    df = pd.DataFrame(rows, columns=["id", "label", "description", "match_text", "aliases", "concepturi"])

    if exact_match_text and not df.empty:
        df = df[df["match_text"].fillna("").str.casefold() == normalized_term.casefold()].reset_index(drop=True)

    if include_detailed_description:
        detailed_descriptions = {}
        if not df.empty:
            detailed_descriptions = fetch_detailed_descriptions_for_entities(
                df["id"].tolist(),
                language=language,
                headers=DEFAULT_HEADERS,
                timeout=30,
                sentences=detailed_description_sentences,
            )
        df.insert(
            df.columns.get_loc("description") + 1,
            "detailed_description",
            [detailed_descriptions.get(entity_id, "") for entity_id in df["id"]],
        )
        if drop_missing_detailed_description:
            df = df[df["detailed_description"].fillna("").str.strip() != ""].reset_index(drop=True)

    if df.empty:
        print(f"No search results for: {term!r}")
    return df


def load_wikidata_definition_candidates(
    query_text: str,
    use_detailed_description: bool = True,
    exact_match_text: bool = False,
) -> tuple[pd.DataFrame, str]:
    candidates_df = search_wikidata(
        query_text,
        limit=5,
        exact_match_text=exact_match_text,
        include_detailed_description=use_detailed_description,
        detailed_description_sentences=3,
        drop_missing_detailed_description=use_detailed_description,
    )

    if candidates_df.empty:
        if use_detailed_description:
            raise ValueError(
                f"search_wikidata returned no detailed_description candidates for span={query_text!r}."
            )
        raise ValueError(
            f"search_wikidata returned no description candidates for span={query_text!r}."
        )

    definition_column = "detailed_description" if use_detailed_description else "description"
    candidates_df = candidates_df[candidates_df[definition_column].fillna("").str.strip() != ""].copy()
    candidates_df = candidates_df.drop_duplicates(subset=["id", definition_column]).reset_index(drop=True)
    if candidates_df.empty:
        raise ValueError(
            f"No usable {definition_column} candidates remained for span={query_text!r}."
        )

    return candidates_df, definition_column


In [6]:
def definition_to_hypothesis(definition: str) -> str:
    cleaned_definition = definition.strip()
    if cleaned_definition.endswith((".", "!", "?")):
        cleaned_definition = cleaned_definition[:-1]
    return f"It refers to {cleaned_definition}."


def extract_cross_encoder_scores(raw_scores, model) -> np.ndarray:
    score_array = np.asarray(raw_scores)
    if score_array.ndim == 1:
        return score_array.astype(float)

    id2label = getattr(model.model.config, "id2label", {}) or {}
    entailment_index = None
    for label_index, label_name in id2label.items():
        if str(label_name).lower() == "entailment":
            entailment_index = int(label_index)
            break

    if entailment_index is None:
        entailment_index = score_array.shape[1] - 1

    return score_array[:, entailment_index].astype(float)


def build_wikidata_candidate_bank(candidates_df: pd.DataFrame, definition_column: str) -> list[dict]:
    candidate_bank = []
    for row in candidates_df.itertuples(index=False):
        definition = str(getattr(row, definition_column)).strip()
        candidate_bank.append(
            {
                "entity_id": row.id,
                "label": row.label,
                "description": row.description,
                "definition_source": definition_column,
                "definition": definition,
                "hypothesis": definition_to_hypothesis(definition),
            }
        )
    return candidate_bank


def evaluate_cross_encoder_on_hotpotqa(
    span_text: str,
    model_name: str = DEFAULT_MODEL_NAME,
    store: dict | None = None,
    kind: str | None = None,
    batch_size: int = DEFAULT_BATCH_SIZE,
    mark_target: bool = False,
    max_records: int | None = None,
    use_detailed_description: bool = True,
    exact_match_text: bool = False,
):
    normalized_span = normalize_text(span_text.strip())
    if not normalized_span:
        raise ValueError("span_text must be a non-empty string.")

    if store is None:
        store = load_hotpot_scan_store()

    records = lookup_records(
        store,
        span_text,
        kind=kind,
        include_text=True,
        include_cleaned_text=True,
    )
    if not records:
        raise ValueError(f"No HotpotQA records were found for span={span_text!r}.")

    if max_records is not None:
        records = records[:max_records]

    candidates_df, definition_column = load_wikidata_definition_candidates(
        span_text.strip(),
        use_detailed_description=use_detailed_description,
        exact_match_text=exact_match_text,
    )
    candidate_bank = build_wikidata_candidate_bank(candidates_df, definition_column=definition_column)
    if not candidate_bank:
        raise ValueError(f"No candidate definitions were built for span={span_text!r}.")

    model = CrossEncoder(model_name)
    evaluation_rows = []

    for record in records:
        prompt_info = build_hotpot_prompt(record, span_text, mark_target=mark_target)
        pairs = [(prompt_info["prompt_text"], candidate["hypothesis"]) for candidate in candidate_bank]
        raw_scores = model.predict(pairs, batch_size=batch_size, show_progress_bar=False)
        scores = extract_cross_encoder_scores(raw_scores, model)

        ranked_candidates = sorted(
            [
                {
                    **candidate,
                    "score": float(score),
                }
                for candidate, score in zip(candidate_bank, scores)
            ],
            key=lambda item: item["score"],
            reverse=True,
        )
        top_candidate = ranked_candidates[0]

        evaluation_rows.append(
            {
                "query_span": span_text,
                "normalized_query": normalized_span,
                "title": record["title"],
                "kind": record["kind"],
                "document_idx": record["document_idx"],
                "span": tuple(record["span"]),
                "matched_text": prompt_info["matched_text"],
                "source_text": prompt_info["context_text"],
                "document_text": record["text"],
                "predicted_entity_id": top_candidate["entity_id"],
                "predicted_label": top_candidate["label"],
                "predicted_description": top_candidate["description"],
                "definition_source": top_candidate["definition_source"],
                "predicted_definition": top_candidate["definition"],
                "prediction_score": top_candidate["score"],
                "prompt_text": prompt_info["prompt_text"],
            }
        )

    results_df = pd.DataFrame(evaluation_rows).sort_values(
        ["prediction_score", "title", "document_idx"],
        ascending=[False, True, True],
    ).reset_index(drop=True)
    return candidates_df, results_df


def display_hotpot_predictions(results_df: pd.DataFrame, limit: int = 20):
    label_counts_df = (
        results_df.groupby(["predicted_entity_id", "predicted_label", "predicted_description"], dropna=False)
        .size()
        .reset_index(name="sample_count")
        .sort_values(["sample_count", "predicted_entity_id"], ascending=[False, True])
        .reset_index(drop=True)
    )
    total_samples = int(len(results_df))

    print(f"Total samples: {total_samples}")
    display(label_counts_df)

    columns = [
        "title",
        "kind",
        "matched_text",
        "source_text",
        "predicted_label",
        "predicted_definition",
        "prediction_score",
    ]
    display(results_df[columns].head(limit))


In [18]:
TARGET_SPAN = "japanese"
QUERY_KIND = None
MARK_TARGET = False
USE_DETAILED_DESCRIPTION = False
EXACT_MATCH_TEXT = True
MAX_RECORDS = 100

candidate_definitions_df, hotpot_results_df = evaluate_cross_encoder_on_hotpotqa(
    span_text=TARGET_SPAN,
    model_name=DEFAULT_MODEL_NAME,
    store=embedding_store,
    kind=QUERY_KIND,
    batch_size=DEFAULT_BATCH_SIZE,
    mark_target=MARK_TARGET,
    max_records=MAX_RECORDS,
    use_detailed_description=USE_DETAILED_DESCRIPTION,
    exact_match_text=EXACT_MATCH_TEXT,
)


In [19]:
definition_column = "detailed_description" if USE_DETAILED_DESCRIPTION else "description"
display(candidate_definitions_df[["id", "label", "description", definition_column]])
display_hotpot_predictions(hotpot_results_df, limit=MAX_RECORDS)
hotpot_results_df.head(100)


,id,label,description,description
0,Q5287,Japanese,language spoken in East Asia,language spoken in East Asia


Total samples: 100


,predicted_entity_id,predicted_label,predicted_description,sample_count
0,Q5287,Japanese,language spoken in East Asia,100


,title,kind,matched_text,source_text,predicted_label,predicted_definition,prediction_score
0,Yunho,phrase,japanese,"fluent in both korean and japanese, yunho has ...",Japanese,language spoken in East Asia,0.202949
1,Wafangdian,phrase,japanese,Wafangdian Bearing Factory is the largest bear...,Japanese,language spoken in East Asia,-0.727792
2,Toys in the Attic (2009 film),phrase,japanese,"it is an international co-production of czech,...",Japanese,language spoken in East Asia,-0.874660
3,Dr. Wily,phrase,Japanese,"In Japanese, he is voiced by Takeshi Aono in a...",Japanese,language spoken in East Asia,-0.879613
4,Christian Bale filmography,phrase,japanese,"bales role of a young boy, interned in china b...",Japanese,language spoken in East Asia,-0.880077
...,...,...,...,...,...,...,...
95,Min'yō,phrase,japanese,the term minyō is now sometimes also used to r...,Japanese,language spoken in East Asia,-3.030396
96,Min'yō,phrase,japanese,the term minyō is now sometimes also used to r...,Japanese,language spoken in East Asia,-3.030396
97,Toyota Innova,phrase,japanese,The Toyota Innova (japanese: トヨタイノーバ toyota in...,Japanese,language spoken in East Asia,-3.216546
98,Mitsubishi Motors,phrase,japanese,Mitsubishi Motors Corporation (japanese: 三菱自動車...,Japanese,language spoken in East Asia,-3.484095


,query_span,normalized_query,title,kind,document_idx,span,matched_text,source_text,document_text,predicted_entity_id,predicted_label,predicted_description,definition_source,predicted_definition,prediction_score,prompt_text
0,japanese,japanese,Yunho,phrase,3688,"(411, 419)",japanese,"fluent in both korean and japanese, yunho has ...",Jung Yun-ho (Hangul: 정윤호 ; Hanja: 鄭允浩 ; born F...,Q5287,Japanese,language spoken in East Asia,description,language spoken in East Asia,0.202949,"Sentence: fluent in both korean and japanese, ..."
1,japanese,japanese,Wafangdian,phrase,14716,"(360, 368)",japanese,Wafangdian Bearing Factory is the largest bear...,"Wafangdian (), formerly Fuxian or Fu County ()...",Q5287,Japanese,language spoken in East Asia,description,language spoken in East Asia,-0.727792,Sentence: Wafangdian Bearing Factory is the la...
2,japanese,japanese,Toys in the Attic (2009 film),phrase,2801,"(443, 451)",japanese,"it is an international co-production of czech,...",Toys in the Attic (Czech: Na půdě aneb Kdo má ...,Q5287,Japanese,language spoken in East Asia,description,language spoken in East Asia,-0.874660,Sentence: it is an international co-production...
3,japanese,japanese,Dr. Wily,phrase,11582,"(264, 272)",Japanese,"In Japanese, he is voiced by Takeshi Aono in a...","Doctor Wily (Dr.ワイリー , Dokutā Wairi ) , ( ) fu...",Q5287,Japanese,language spoken in East Asia,description,language spoken in East Asia,-0.879613,"Sentence: In Japanese, he is voiced by Takeshi..."
4,japanese,japanese,Christian Bale filmography,phrase,3705,"(349, 357)",japanese,"bales role of a young boy, interned in china b...",British actor Christian Bale has starred in va...,Q5287,Japanese,language spoken in East Asia,description,language spoken in East Asia,-0.880077,"Sentence: bales role of a young boy, interned ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,japanese,japanese,Min'yō,phrase,12071,"(623, 631)",japanese,the term minyō is now sometimes also used to r...,Min'yō (民謡 ) is a genre of traditional Japanes...,Q5287,Japanese,language spoken in East Asia,description,language spoken in East Asia,-3.030396,Sentence: the term minyō is now sometimes also...
96,japanese,japanese,Min'yō,phrase,12071,"(697, 705)",japanese,the term minyō is now sometimes also used to r...,Min'yō (民謡 ) is a genre of traditional Japanes...,Q5287,Japanese,language spoken in East Asia,description,language spoken in East Asia,-3.030396,Sentence: the term minyō is now sometimes also...
97,japanese,japanese,Toyota Innova,phrase,18523,"(19, 27)",japanese,The Toyota Innova (japanese: トヨタイノーバ toyota in...,"The Toyota Innova (Japanese: トヨタイノーバ ""Toyota I...",Q5287,Japanese,language spoken in East Asia,description,language spoken in East Asia,-3.216546,Sentence: The Toyota Innova (japanese: トヨタイノーバ...
98,japanese,japanese,Mitsubishi Motors,phrase,7487,"(31, 39)",japanese,Mitsubishi Motors Corporation (japanese: 三菱自動車...,Mitsubishi Motors Corporation (Japanese: 三菱自動車...,Q5287,Japanese,language spoken in East Asia,description,language spoken in East Asia,-3.484095,Sentence: Mitsubishi Motors Corporation (japan...
